# SafeTour AI: Road Network Safety Prediction using Graph Attention Networks (GATv2)
### IEEE Paper Research Contribution: Edge-Conditioned Multi-Head GAT on Real Geospatial Road Graphs

This notebook trains an **Edge-Conditioned Graph Attention Network (GATv2)** on the real OpenStreetMap road network graph of **Krishnankoil / Srivilliputtur, Tamil Nadu**, enriched with real Tamil Nadu road accident statistics and police complaint distributions.

---

## 1. Setup & Dependencies

In [ ]:
# Install dependencies for Google Colab
!pip install torch-geometric osmnx networkx scikit-learn pandas numpy matplotlib seaborn

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch_geometric.data import Data
from torch_geometric.nn import GATv2Conv
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Compute device:', device)

## 2. Load Featured Road Graph & Construct PyG Tensors

In [ ]:
# Upload or locate nodes.csv and edges.csv
nodes_path = 'data/nodes.csv' if os.path.exists('data/nodes.csv') else '../data/nodes.csv'
edges_path = 'data/edges.csv' if os.path.exists('data/edges.csv') else '../data/edges.csv'

df_nodes = pd.read_csv(nodes_path)
df_edges = pd.read_csv(edges_path)

print(f'Nodes count: {len(df_nodes):,}, Edges count: {len(df_edges):,}')

# Continuous node ID mapping
unique_nids = df_nodes['node_id'].unique()
node_to_idx = {nid: i for i, nid in enumerate(unique_nids)}
num_nodes = len(unique_nids)

# Graph connectivity
u_idx = df_edges['u'].map(node_to_idx).values
v_idx = df_edges['v'].map(node_to_idx).values
edge_index = torch.tensor(np.array([u_idx, v_idx]), dtype=torch.long)

# Node features
lat_s = (df_nodes['lat'].values - df_nodes['lat'].mean()) / (df_nodes['lat'].std() + 1e-6)
lng_s = (df_nodes['lng'].values - df_nodes['lng'].mean()) / (df_nodes['lng'].std() + 1e-6)
log_acc = np.log1p(df_nodes['accident_count'].values)
acc_sev = df_nodes['accident_severity'].values / 1000.0
reg_acc = df_nodes['regional_accidents'].values / 5000.0
crime = df_nodes['crime_score'].values / 100.0
poi = df_nodes['nearby_poi_count'].values.astype(np.float32)
out_deg = np.bincount(u_idx, minlength=num_nodes).astype(np.float32)
in_deg = np.bincount(v_idx, minlength=num_nodes).astype(np.float32)

X = torch.tensor(np.column_stack([
    lat_s, lng_s, log_acc, acc_sev, reg_acc, crime, poi, in_deg, out_deg
]), dtype=torch.float)

# Edge features
ROAD_CLASSES = ['trunk', 'primary', 'secondary', 'tertiary', 'residential', 'service', 'unclassified', 'other']
def encode_rt(rt):
    vec = [0.0]*len(ROAD_CLASSES)
    rt_s = str(rt).lower().strip()
    vec[ROAD_CLASSES.index(rt_s) if rt_s in ROAD_CLASSES else -1] = 1.0
    return vec

norm_len = (np.log1p(df_edges['length'].values) / 6.0).astype(np.float32)
oneway = df_edges['oneway'].astype(np.float32).values
rt_onehot = np.array([encode_rt(x) for x in df_edges['road_type'].values], dtype=np.float32)
e_crime = (df_edges['crime_score'].values / 100.0).astype(np.float32)
e_acc = (df_edges['accident_risk'].values / 100.0).astype(np.float32)
e_flood = (df_edges['flood_risk'].values / 100.0).astype(np.float32)
e_tourist = (df_edges['tourist_density'].values / 100.0).astype(np.float32)
e_blocked = df_edges['road_blocked'].astype(np.float32).values

edge_attr = torch.tensor(np.column_stack([
    norm_len, oneway, rt_onehot, e_crime, e_acc, e_flood, e_tourist, e_blocked
]), dtype=torch.float)

# Target safety score
road_vuln_map = {'trunk': 15.0, 'primary': 20.0, 'secondary': 30.0, 'tertiary': 40.0, 'residential': 50.0, 'service': 65.0, 'unclassified': 55.0, 'other': 60.0}
vuln = df_edges['road_type'].map(lambda x: road_vuln_map.get(str(x).lower().strip(), 50.0)).values
hazard = (0.35 * df_edges['accident_risk'].values + 0.25 * df_edges['crime_score'].values + 0.15 * df_edges['flood_risk'].values + 0.15 * vuln + 0.10 * df_edges['tourist_density'].values)
ground_truth = np.clip(100.0 - hazard, 0.0, 100.0)
Y = torch.tensor(ground_truth, dtype=torch.float).unsqueeze(1)

# Splits: 70% Train, 15% Val, 15% Test
np.random.seed(42)
num_edges = len(df_edges)
perm = np.random.permutation(num_edges)
train_mask = torch.zeros(num_edges, dtype=torch.bool)
val_mask = torch.zeros(num_edges, dtype=torch.bool)
test_mask = torch.zeros(num_edges, dtype=torch.bool)
train_mask[perm[:int(0.70 * num_edges)]] = True
val_mask[perm[int(0.70 * num_edges):int(0.85 * num_edges)]] = True
test_mask[perm[int(0.85 * num_edges):]] = True

data = Data(x=X, edge_index=edge_index, edge_attr=edge_attr, y=Y, train_mask=train_mask, val_mask=val_mask, test_mask=test_mask).to(device)
print('Data tensor prepared on', device)

## 3. RoadSafetyGAT Model Definition

In [ ]:
class RoadSafetyGAT(nn.Module):
    def __init__(self, node_in_dim=9, edge_in_dim=15, hidden_dim=64, num_heads=4, num_layers=2, dropout=0.1):
        super().__init__()
        self.node_encoder = nn.Sequential(
            nn.Linear(node_in_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.LeakyReLU(0.2)
        )
        head_dim = hidden_dim // num_heads
        self.gat_convs = nn.ModuleList([
            GATv2Conv(hidden_dim, head_dim, heads=num_heads, edge_dim=edge_in_dim, concat=True, dropout=dropout, add_self_loops=True)
            for _ in range(num_layers)
        ])
        self.layer_norms = nn.ModuleList([nn.LayerNorm(hidden_dim) for _ in range(num_layers)])
        
        edge_repr_dim = (hidden_dim * 2) + edge_in_dim
        self.edge_predictor = nn.Sequential(
            nn.Linear(edge_repr_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )
        self.dropout = dropout

    def forward(self, x, edge_index, edge_attr, return_attention=False):
        h = self.node_encoder(x)
        att_weights = None
        for i, (conv, norm) in enumerate(zip(self.gat_convs, self.layer_norms)):
            h_in = h
            if return_attention and i == 0:
                h_out, att = conv(h, edge_index, edge_attr=edge_attr, return_attention_weights=True)
                att_weights = att
            else:
                h_out = conv(h, edge_index, edge_attr=edge_attr)
            h = norm(h_in + F.elu(h_out))
            h = F.dropout(h, p=self.dropout, training=self.training)
        
        u_idx, v_idx = edge_index[0], edge_index[1]
        edge_repr = torch.cat([h[u_idx], h[v_idx], edge_attr], dim=-1)
        pred_safety = 100.0 * torch.sigmoid(self.edge_predictor(edge_repr))
        if return_attention:
            return pred_safety, att_weights
        return pred_safety

model = RoadSafetyGAT(node_in_dim=9, edge_in_dim=15, hidden_dim=64, num_heads=4).to(device)
print(f'Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

## 4. Train Model with Validation & Early Stopping

In [ ]:
optimizer = AdamW(model.parameters(), lr=0.003, weight_decay=1e-4)
criterion = nn.SmoothL1Loss(beta=1.0)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=10, min_lr=1e-5)

epochs = 120
best_val_rmse = float('inf')
history = {'train_loss': [], 'val_rmse': [], 'val_mae': []}

for epoch in range(1, epochs + 1):
    model.train()
    optimizer.zero_grad()
    pred = model(data.x, data.edge_index, data.edge_attr)
    loss = criterion(pred[data.train_mask], data.y[data.train_mask])
    loss.backward()
    nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0)
    optimizer.step()

    # Validation
    model.eval()
    with torch.no_grad():
        val_pred = model(data.x, data.edge_index, data.edge_attr)
        y_val_t = data.y[data.val_mask].cpu().numpy().flatten()
        y_val_p = val_pred[data.val_mask].cpu().numpy().flatten()
        val_rmse = np.sqrt(mean_squared_error(y_val_t, y_val_p))
        val_mae = mean_absolute_error(y_val_t, y_val_p)

    scheduler.step(val_rmse)
    history['train_loss'].append(loss.item())
    history['val_rmse'].append(val_rmse)
    history['val_mae'].append(val_mae)

    if val_rmse < best_val_rmse:
        best_val_rmse = val_rmse
        torch.save(model.state_dict(), 'best_gat_model.pt')

    if epoch % 10 == 0 or epoch == 1:
        print(f'Epoch {epoch:03d} | Train Loss: {loss.item():.4f} | Val RMSE: {val_rmse:.2f} | Val MAE: {val_mae:.2f}')

print(f'Done! Best Val RMSE: {best_val_rmse:.2f}')

## 5. Test Evaluation & IEEE Publication Plots

In [ ]:
# Load best weights
model.load_state_dict(torch.load('best_gat_model.pt', map_location=device))
model.eval()
with torch.no_grad():
    test_pred, att_weights = model(data.x, data.edge_index, data.edge_attr, return_attention=True)
    y_true_test = data.y[data.test_mask].cpu().numpy().flatten()
    y_pred_test = test_pred[data.test_mask].cpu().numpy().flatten()

test_rmse = np.sqrt(mean_squared_error(y_true_test, y_pred_test))
test_mae = mean_absolute_error(y_true_test, y_pred_test)
test_r2 = r2_score(y_true_test, y_pred_test)

print(f'Test RMSE: {test_rmse:.3f}')
print(f'Test MAE:  {test_mae:.3f}')
print(f'Test R²:   {test_r2:.4f}')

# Plotting
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. Loss curve
axes[0].plot(history['train_loss'], label='Train Loss (Huber)', color='#1E88E5')
axes[0].plot(history['val_rmse'], label='Val RMSE', color='#D81B60')
axes[0].set_title('Training Convergence')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Metric')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 2. Parity plot
axes[1].scatter(y_true_test, y_pred_test, alpha=0.4, color='#004D40', edgecolors='none')
axes[1].plot([50, 85], [50, 85], 'r--', label='Ideal Fit')
axes[1].set_title(f'Actual vs Predicted Safety Score (R²={test_r2:.3f})')
axes[1].set_xlabel('Ground Truth Safety Score')
axes[1].set_ylabel('GATv2 Predicted Safety Score')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()